In [2]:
import timm 


timm.list_models("darknet*")

['darknet17', 'darknet21', 'darknet53', 'darknetaa53']

In [4]:
models = ["resnet18", "darknet17", "convnext_nano"]

for model_name in models:
    model = timm.create_model(model_name, pretrained=False)

    # 2. Count total parameters (including frozen layers)
    total_params = sum(p.numel() for p in model.parameters())

    # 3. Count trainable parameters only (where requires_grad=True)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(model_name)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print("================================================================")

resnet18
Total Parameters: 11,689,512
Trainable Parameters: 11,689,512
darknet17
Total Parameters: 14,304,840
Trainable Parameters: 14,304,840
convnext_nano
Total Parameters: 15,593,560
Trainable Parameters: 15,593,560


In [ ]:
# main nets
""" 
resnet
darknet
convnext
"""

# hypernets
""" 
resnet
convnext
swin_transformer
"""

In [41]:
models = ["resnet200", "convnext_base", "swin_base_patch4_window12_384"]

for model_name in models:
    model = timm.create_model(model_name, pretrained=False)

    # 2. Count total parameters (including frozen layers)
    total_params = sum(p.numel() for p in model.parameters())

    # 3. Count trainable parameters only (where requires_grad=True)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(model_name)
    print(f"Total Parameters: {total_params:,}")
    print(f"Trainable Parameters: {trainable_params:,}")
    print("================================================================")

resnet200
Total Parameters: 64,673,832
Trainable Parameters: 64,673,832
convnext_base
Total Parameters: 88,591,464
Trainable Parameters: 88,591,464
swin_base_patch4_window12_384
Total Parameters: 87,903,584
Trainable Parameters: 87,903,584


In [9]:
import torch
import torchviz

s = 480
# Загружаем модель
model = timm.create_model("swin_base_patch4_window12_384", pretrained=False, img_size=s)

# Создаем dummy input
dummy_input = torch.randn(1, 3, s, s)

# Визуализируем
output = model(dummy_input)
# graph = torchviz.make_dot(output, params=dict(list(model.named_parameters()) + [('input', dummy_input)]))
# graph.render("model_architecture", format="png")  # Сохранит в PNG
# graph.view() 

In [11]:
model = timm.create_model("resnet200", pretrained=False)

In [4]:
import math 
from torch import nn 

class WeightsGeneratorLinearHead(nn.Module):
    def __init__(self, model: nn.Module, input_size):
        super().__init__()

        self.heads = nn.ModuleDict()
        self.head_linear_params = dict()

        for pp in model.named_parameters():
            layer_name = pp[0].replace(".", "_")
            layer_shape = list(pp[1].data.shape)
            out_size = math.prod(layer_shape)

            self.heads[layer_name] = nn.Linear(input_size, out_size)
            self.head_linear_params[layer_name] = layer_shape 

    def forward(self, x):
        return {name: head(x) for name, head in self.heads.items()}


In [8]:
list(model.modules())[-2]

Linear(in_features=1024, out_features=1000, bias=True)

In [9]:
# ["resnet18", "darknet17", "convnext_nano"]

model = timm.create_model("darknet17", pretrained=False)

in_feat = 1024
heads = WeightsGeneratorLinearHead(model, in_feat)



RuntimeError: [enforce fail at alloc_cpu.cpp:117] data. DefaultCPUAllocator: not enough memory: you tried to allocate 4194304000 bytes.

In [3]:
import math

model = timm.create_model("resnet200", pretrained=False)

c = 0 
for pp in model.named_parameters():
    print(pp[0])
    print(pp[1].data.shape)
    size = list(pp[1].data.shape)
    print(math.prod(size))
    c+= 1

c

conv1.weight
torch.Size([64, 3, 7, 7])
9408
bn1.weight
torch.Size([64])
64
bn1.bias
torch.Size([64])
64
layer1.0.conv1.weight
torch.Size([64, 64, 1, 1])
4096
layer1.0.bn1.weight
torch.Size([64])
64
layer1.0.bn1.bias
torch.Size([64])
64
layer1.0.conv2.weight
torch.Size([64, 64, 3, 3])
36864
layer1.0.bn2.weight
torch.Size([64])
64
layer1.0.bn2.bias
torch.Size([64])
64
layer1.0.conv3.weight
torch.Size([256, 64, 1, 1])
16384
layer1.0.bn3.weight
torch.Size([256])
256
layer1.0.bn3.bias
torch.Size([256])
256
layer1.0.downsample.0.weight
torch.Size([256, 64, 1, 1])
16384
layer1.0.downsample.1.weight
torch.Size([256])
256
layer1.0.downsample.1.bias
torch.Size([256])
256
layer1.1.conv1.weight
torch.Size([64, 256, 1, 1])
16384
layer1.1.bn1.weight
torch.Size([64])
64
layer1.1.bn1.bias
torch.Size([64])
64
layer1.1.conv2.weight
torch.Size([64, 64, 3, 3])
36864
layer1.1.bn2.weight
torch.Size([64])
64
layer1.1.bn2.bias
torch.Size([64])
64
layer1.1.conv3.weight
torch.Size([256, 64, 1, 1])
16384
layer1.

611

In [2]:
import timm

m = timm.create_model("resnet18", pretrained=False)

m.fc

Linear(in_features=512, out_features=1000, bias=True)